In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from collections import Counter
import string
import time
import copy
import random
import json

import preprocess
import model_arch

In [2]:
def get_device():
    if torch.backends.mps.is_available():
        print("MPS is available")
        return torch.device("mps")
    elif torch.cuda.is_available():
        print("CUDA is available")
        return torch.device("cuda")
    else:
        print("CPU is available")
        return torch.device("cpu")

In [ ]:
DEVICE = get_device() # torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print('Using device:', str(DEVICE).upper(), "\n")

In [ ]:
#  ---- Run this code for the first model initialization ---- 

#   Model configuration structure
#   {
#       "embedding_shape": < model.embedding.weight.data.shape >,
#       "window_size": < WINDOW_SIZE >,
#       "batch_size: < BATCH_SIZE >,
#   
#       # Path to the model
#       "model_path": "model/< MODEL_NAME >.pt",
#
#       "word2idx": { ... },
#   }

"""
# EMBEDDING_SIZE = 256
# WINDOW_SIZE = 2
# BATCH_SIZE = 128
# DATA_PATH = "data/comb.txt"

## word2idx is a vocabulary (vocab) that has the format {"word_1: idx_1, "word_2", idx_2, ...}
word2idx, idx2word = preprocess.get_index(DATA_PATH)

sequence, vocab_size = preprocess.get_sequence(DATA_PATH)
dataset = model_arch.CBOWDataset(sequence, WINDOW_SIZE)
model = model_arch.CBOWModel(vocab_size, EMBEDDING_SIZE)

## Save model configuration in the JSON file

hash_5 = ''.join(random.choices(string.ascii_letters + string.digits, k=5))
model_params = {
    "embedding_shape": tuple(model.embedding.weight.data.shape),
    "window_size": WINDOW_SIZE,
    "batch_size": BATCH_SIZE,
    "model_path": f"model/{MODEL_NAME}_{hash_5}.pt",
    "word2idx": word2idx,
}
with open(f"model/{MODEL_NAME}.json", "w") as f:
    json.dump(model_params, f, indent=4)
"""

In [ ]:
# --- Load pretrained model from the data and model file *.pt
# --- Run this code only if there is a saved pretrained model

# Read the model configuration from the file and parse it
with open(f"model/model_config_v2.json", "r") as f:
    model_params = json.load(f)

# Define model params
embedding_size = model_params['embedding_size']
window_size    = model_params['window_size']
batch_size     = model_params['batch_size']
model_path     = model_params['model_path']
sequence       = torch.tensor(model_params['sequence'])
word2idx       = model_params['word_to_idx']

vocab_size = len(word2idx)
idx2word = {}
for k, v in word2idx.items():
    idx2word[v] = k

# sequence, vocab_size = preprocess.get_sequence("data/comb.txt")
# dataset = model_arch.CBOWDataset(sequence, window_size)
# word2idx, idx2word = preprocess.get_index(data_path)

dataset = model_arch.CBOWDataset(sequence, window_size)
model = model_arch.CBOWModel(vocab_size, embedding_size)
try:
    model.load_state_dict(torch.load(model_path))
    
    print(f" -- Model nn.Embedding shape: {model.embedding.weight.data.shape}")
    print(f" -- Model nn.Linear shape: {model.fc.weight.data.shape}")
except Exception as error:
    print(f"An exception of type {type(error).__name__} occurred: {error}")

In [ ]:
### --- MODEL RESIZING ---
### Run this code only if different data is used to learn the model

DATA_PATH = "data/war_and_peace.txt"

print(" -- Initial vocab size:", len(word2idx))
# Save values
new_vocab = copy.deepcopy(word2idx)
last_value = len(new_vocab)

boof_vocab, _ = preprocess.get_index(DATA_PATH)
boof_sequence, _ = preprocess.get_sequence(DATA_PATH)

for k, v in boof_vocab.items():
    if new_vocab.get(k, None) == None:
        # print(k, " --- ", last_value)
        new_vocab[k] = last_value
        last_value += 1

new_vocab_size = len(new_vocab)
print(" -- New vocab size:", new_vocab_size)


if len(new_vocab) > len(word2idx):
    model.embedding = preprocess.resize_embedding(model.embedding, len(new_vocab))

    # This resizing affects only to linear part of weights and do not use.
    # Do we need to use it???
    model.fc = preprocess.resize_linear(model.fc, len(new_vocab))

    # Update 'sequence' with new values
    model_params['sequence'].extend(boof_sequence.tolist())

    # Update 'vocab' with new key-values
    print(len(new_vocab))
    model_params['word_to_idx'] = new_vocab

else:
    print(" -- Skipping model resizing, new vocab does not contain new words")

In [ ]:
### --- MODEL TRAINING ---
model = model.to(DEVICE)

epochs = 2
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

for epoch in range(epochs):
    start = time.perf_counter()
    
    total_loss = 0
    for context, target in dataloader:
        context, target = context.to(DEVICE), target.to(DEVICE) # move to GPU
        
        optimizer.zero_grad()
        logits = model(context)
        loss = criterion(logits, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    end = time.perf_counter()

    print(f" ---- Epoch {epoch+1}, Loss: {total_loss:.4f}, Execution time: {end - start:.6f}")

In [ ]:
### --- Save model ---
model = model.to("cpu")
print(f" ---- Model shape is: {model.embedding.weight.data.shape}")
torch.save(model.state_dict(), f"model/model_cpu_v2.pt")

model_params['model_path'] = "model/model_cpu_v2.pt"
print(f" ---- Model has been saved in file model_cpu_v2.pt")

# Save model in the file
with open(f"model/model_config_v2.json", "w") as f:
    json.dump(model_params, f, indent=4)

In [ ]:
model = model.to(DEVICE)
word_embeddings = model.embedding.weight.detach()

# Example prediction
sentense = "There are two sides to".lower()
context_words = []

split_sentense = sentense.split(" ")

if (len(split_sentense) - 1) != window_size * 2:
    print("ERROR: sentense does not have the сorrect lenght for testinng")
else:
    for w in split_sentense:
        context_words.append(w)
    del context_words[window_size]

print("Context words: ", context_words)

In [ ]:
top_k = 5
model = model.to(DEVICE)
model.eval()

idx2word = {}
for k, v in word2idx.items():
    idx2word[v] = k

context_idxs = torch.tensor(
    [[word2idx[w] for w in context_words]]
)
context_idxs = context_idxs.to(DEVICE)

with torch.no_grad():
    logits = model(context_idxs)
    probs = torch.softmax(logits, dim=1)

top_idxs = torch.topk(probs, top_k).indices[0]
top_idxs
result = [idx2word[idx.item()] for idx in top_idxs]
print(result)

In [ ]:
word_1 = "he"
word_2 = "she"
threshold = 0.01

v1 = word_embeddings[word2idx[word_1]]
v2 = word_embeddings[word2idx[word_2]]
print(f"Original: {F.cosine_similarity(v1, v2, dim=0).item()}")


def convert(arr):
    new_arr = []
    for i in arr:
        if abs(i) >= threshold:
            new_arr.append(0.0)
        else:
            new_arr.append(1.0)
    return torch.tensor(new_arr)


idx1 = word2idx[word_1]
idx2 = word2idx[word_2]

word_emb1 = convert(word_embeddings[idx1])
word_emb2 = convert(word_embeddings[idx2])

print(f"Filtered: {F.cosine_similarity(word_emb1, word_emb2, dim=0).item()}")

In [ ]:
coincides = 0
w1_pos = 0
w2_pos = 0
for i in range(len(word_emb1)):
    if word_emb1[i] == 1.0:
        w1_pos += 1
    if word_emb2[i] == 1.0:
        w2_pos += 1
    if word_emb1[i] == 1.0 and word_emb2[i] == 1.0:
        coincides += 1

print(f"Number of 1-position for {word_1}: {w1_pos}")
print(f"Number of 1-position for {word_2}: {w2_pos}")
print(f"Number of coincides: {coincides}")

In [ ]:
vision = ["vision", "color", "red", "orange", "yellow", "green", "blue", "violet", "purple", "lilac"]
teste  = ["taste", "bitter", "sweet", "sour"]

for w in teste:
    print(w, " -----> ", word_embeddings[word2idx[word_1]].tolist())

In [ ]:
x = list(range(256))
plt.figure(figsize=(32, 16))

words = ["bitter", "sweet"]
for w in words:
    plt.plot(x, word_embeddings[word2idx[w]].tolist(), label=w)

plt.xlabel("X-axis")
plt.ylabel("Y-axis")
plt.legend()  # shows labels for each line
plt.ylim(-25, 25)
plt.grid(True)

# Save as image
plt.savefig("plot_5.png")

plt.show()